# Entrenamiento del clasificador de estados de tráfico

Usá esta notebook para crear el bundle piloto inicial o reentrenar el MLP con evidencia HITL. El preset selecciona una receta completa; el objeto validado sigue siendo la única fuente de verdad.

| Preset | Entrada principal | Holdout |
|---|---|---|
| `TrainingPreset.SEED_UPLOAD` | Backup o CSV subido | Provisional |
| `TrainingPreset.SEED_POSTGRES` | PostgreSQL raw read-only | Provisional |
| `TrainingPreset.HITL_CATALOG` | Semilla + catálogo HITL | Provisional |
| `TrainingPreset.HITL_CATALOG_POSTGRES` | Catálogo + PostgreSQL | Provisional |
| `TrainingPreset.HITL_FROZEN_HOLDOUT` | Catálogo HITL | Humano congelado |
| `TrainingPreset.CUSTOM` | Un preset base con cambios explícitos | Configurable |

**Inicio rápido recomendado:** para el backup histórico conservá `TrainingPreset.SEED_UPLOAD`, ejecutá `Run All` y cargá `traffic_data.backup` cuando aparezca el selector.

<details>
<summary><strong>Personalización y operaciones excepcionales</strong></summary>

Usá `CUSTOM_CONFIG = replace(training_preset_config(...), ...)` para habilitar validación cruzada, comparar una corrida o crear una nueva generación con motivo. La copia del bundle permanece desactivada. Consultá la [guía central de presets](../../../docs/operations/notebook-configuration.md).

</details>


In [ ]:
# Preparación del entorno: ejecutá esta celda una vez por runtime.
import importlib.util
import os
import runpy
import subprocess
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
WORKSPACE_DIR = Path("/content/vaaet")
if IN_COLAB:
    if (WORKSPACE_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(WORKSPACE_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(WORKSPACE_DIR)])
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    WORKSPACE_DIR = next(
        (
            path
            for path in candidates
            if (path / "vaaet-core/pyproject.toml").is_file()
            and (path / "vaaet-persistence/pyproject.toml").is_file()
            and (path / "vaaet-ml/pyproject.toml").is_file()
        ),
        None,
    )
    if WORKSPACE_DIR is None:
        raise RuntimeError("No se encontró el workspace VAAET con core y ML.")
CORE_ROOT = WORKSPACE_DIR / "vaaet-core"
PERSISTENCE_ROOT = WORKSPACE_DIR / "vaaet-persistence"
ML_ROOT = WORKSPACE_DIR / "vaaet-ml"
REPO_ROOT = ML_ROOT
os.chdir(ML_ROOT)
BOOTSTRAP = runpy.run_path(str(ML_ROOT / "scripts" / "notebook_bootstrap.py"))
RUNTIME = BOOTSTRAP["bootstrap_notebook"](
    workspace_root=WORKSPACE_DIR,
    core_root=CORE_ROOT,
    persistence_root=PERSISTENCE_ROOT,
    ml_root=ML_ROOT,
    core_extras=('inference',),
    ml_extras=('training', 'visualization', 'database'),
    in_colab=IN_COLAB,
    framework='tensorflow',
    require_gpu=True,
)
VAAET_PACKAGE_FILE = RUNTIME.package_file
VAAET_ML_PACKAGE_FILE = RUNTIME.ml_package_file
GIT_COMMIT = RUNTIME.git_commit


In [ ]:
# Configuración del workflow: editá únicamente esta celda.
from dataclasses import replace

from vaaet_ml.workflow_presets import (
    TrainingPreset,
    render_workflow_summary,
    resolve_training_config,
    training_preset_config,
)

SELECTED_PRESET = TrainingPreset.SEED_UPLOAD
CUSTOM_CONFIG = None
SEED_RAW_SOURCE_PATH: Path | None = None  # .backup o .csv; None abre el selector en Colab.
WORKFLOW_CONFIG = resolve_training_config(
    SELECTED_PRESET,
    custom_config=CUSTOM_CONFIG,
)

print(render_workflow_summary(SELECTED_PRESET, WORKFLOW_CONFIG))


In [ ]:
# Imports del workflow: no edites esta celda.
import os
import shutil
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import sqlalchemy
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from vaaet.artifacts import FEATURE_SCHEMA_VERSION
from vaaet_persistence import PipelineRunMetadata, PipelineWorkflow, dispose_engine, finish_pipeline_run, get_engine, start_pipeline_run
from vaaet_ml.data.database import DatabaseProfile, get_optional_database_settings
from vaaet_ml.data.postgres_restore import resolve_pg_restore_for_backup
from vaaet_ml.data.dataset_artifacts import CatalogSelection, DatasetArtifactAction, HitlCatalogSource, SeedArtifactConfig, VersionedSeedStore, create_training_input_lock
from vaaet_ml.data.ingestion import FeedbackPolicy, PostgresBackupSource, PostgresSource, RawCsvSource, SeedDatasetPackageSource, TrainingIngestionPlan, compose_supervised_dataset, load_training_inputs
from vaaet_ml.data.datasets import build_group_ids
from vaaet.timestamps import normalize_timestamp_series
from vaaet.calibration import apply_temperature_scaling, fit_temperature, multiclass_brier_score
from vaaet_ml.evaluation.dataset_validation import audit_training_dataset
from vaaet_ml.evaluation.reporting import build_class_support_notes, build_classification_support_table, expected_calibration_error, expected_confusion_cost, false_alert_rate_upper_bound, grouped_classification_intervals, plot_training_evaluation, plot_training_history, select_validation_decision_policy, summarize_data_origin, summarize_state_balance
from vaaet.features.engineering import engineer_features
from vaaet.features.labeling import assign_stable_traffic_state
from vaaet_ml.features.synthetic import augment_with_synthetic
from vaaet.inference.traffic_state import apply_conservative_accident_gate, classify_telemetry_dataframe
from vaaet.logging import configure_logging
from vaaet_ml.settings import DATA_PROCESSED_DIR, DATA_RAW_DIR, DRIVE_ARTIFACT_DIR, FEATURE_COLS, LABELING_THRESHOLDS, MODEL_DIR, MODEL_VERSION, N_MODEL_STATES, RANDOM_SEED, STATE_LABELS
from vaaet_ml.training.balancing import BalanceStrategy, build_balance_candidates, compute_capped_balanced_weights
from vaaet_ml.training.bundle_export import build_and_publish_bundle, publish_bundle_copy
from vaaet_ml.training.holdout import HumanHoldoutAction, HumanHoldoutConfig, resolve_human_holdout
from vaaet_ml.training.lifecycle import ModelInputPolicy, TrainingMode, apply_model_input_policy, build_supervision_weights, build_training_lifecycle, cap_synthetic_congested_weight
from vaaet_ml.training.modeling import build_traffic_state_mlp
from vaaet_ml.training.partitions import build_training_partitions
from vaaet_ml.training.selection import select_balance_candidate

configure_logging()
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
ACTIVE_TRAINING_MODE = TrainingMode(WORKFLOW_CONFIG.training_mode)


In [ ]:
# Servicios de entrenamiento: no edites esta celda.
import json
from importlib.metadata import version as package_version

from vaaet_ml.evaluation.reporting import save_training_run_diagnostics
from vaaet_ml.runtime import build_training_runtime_evidence
from vaaet_ml.training.cross_validation import run_grouped_cross_validation
from vaaet_ml.training.eligibility import evaluate_candidate_eligibility
from vaaet_ml.training.execution import (
    TrainingFitConfig,
    build_training_callbacks,
    set_keras_random_seed,
)
from vaaet_ml.training.observability import (
    TRAINING_OBSERVABILITY_REPORT_FILE,
    build_training_evaluation_evidence,
    build_training_run_report,
    compare_training_run_reports,
    load_training_run_report,
    write_training_run_report,
)


## 1. Entender los dos modos

- **Inicio Semilla (`SEED_BOOTSTRAP`)**: toma datos crudos, calcula las 19 features y crea etiquetas provisionales mediante reglas. Es el punto de partida rápido.
- **Reentrenamiento HITL (`HITL_RETRAINING`)**: toma features ya calculadas y correcciones humanas. No vuelve a hacer ingeniería sobre esos registros.

<details>
<summary><strong>Diccionario en palabras simples</strong></summary>

- **Weak supervision:** reglas que crean etiquetas provisionales cuando todavía no hay suficientes revisiones humanas.
- **Memoria proxy decreciente:** al principio conserva parte de la semilla; a medida que llegan etiquetas humanas, esa influencia baja por clase hasta desaparecer.
- **Datos sintéticos:** ejemplos artificiales usados sólo en train para ejercitar congestión e incidentes; nunca prueban calidad real.
- **Holdout humano:** un examen fijo que el modelo nunca usa para aprender. Permite comparar candidatos con las mismas preguntas.
- **Snapshot semilla:** fotografía procesada e inmutable de los datos iniciales.
- **Catálogo HITL:** lista verificada de paquetes producidos por sesiones de revisión.
- **Training input lock:** comprobante exacto de qué semilla, feedback y holdout usó un entrenamiento.
- **Pilot / candidate / production:** piloto sirve para iniciar el ciclo; candidato espera evaluación; producción superó todos los gates.

</details>

Las fuentes son explícitas: backup/CSV/PostgreSQL para raw; snapshot semilla y catálogo/PostgreSQL para HITL. Nunca se adivina el tipo por sus columnas y una predicción sin revisar jamás se convierte en etiqueta.

PostgreSQL es siempre read-only y sólo se consulta con el preset PostgreSQL correspondiente. Configurá el perfil `training` según la [guía canónica de Colab](../../../docs/operations/colab-guide.md#secrets-y-postgresql).

## 2. Preparar archivos y fuentes

En modo semilla, Drive guarda el snapshot inmutable y Colab solicita el backup o CSV sólo cuando hace falta. En modo HITL, se reutilizan la semilla y el catálogo de revisiones de Drive. La siguiente etapa muestra exactamente qué fuente encontró.

In [ ]:
# En Colab, declarás una fuente permitida sólo cuando el preset la requiere.
# HITL lee el catálogo inmutable de Drive; la semilla se resuelve por current.json.
# En local, usá las raíces equivalentes respaldadas por el filesystem.
_backup_dest = os.path.join(DATA_RAW_DIR, "traffic_data.backup")
_csv_dest = os.path.join(DATA_RAW_DIR, "traffic_data_raw.csv")
HUMAN_HOLDOUT_STORE_ROOT = DATA_PROCESSED_DIR / "holdouts"
SEED_ARTIFACT_ROOT = DATA_PROCESSED_DIR / "seed-bootstrap"
HITL_CATALOG_PATH = DATA_PROCESSED_DIR / "hitl-reviews/catalog.json"
TRAINING_RUNS_ROOT = DATA_PROCESSED_DIR / "training-runs"
if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore[import-untyped]
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        raise RuntimeError("Los datasets inmutables necesitan Google Drive montado; no se usará un fallback efímero.") from exc
    _drive_data_root = Path("/content/drive/MyDrive/vaaet-ml/data")
    SEED_ARTIFACT_ROOT = _drive_data_root / "seed-bootstrap"
    HITL_CATALOG_PATH = _drive_data_root / "hitl-reviews/catalog.json"
    TRAINING_RUNS_ROOT = Path("/content/drive/MyDrive/vaaet-ml/training-runs")
    HUMAN_HOLDOUT_STORE_ROOT = Path("/content/drive/MyDrive/vaaet-ml/data/holdouts")
    for _artifact_directory in (SEED_ARTIFACT_ROOT, HITL_CATALOG_PATH.parent, TRAINING_RUNS_ROOT, HUMAN_HOLDOUT_STORE_ROOT):
        _artifact_directory.mkdir(parents=True, exist_ok=True)
    print(f"🔒 Datos inmutables en Drive: {_drive_data_root}")

RESOLVED_SEED_RAW_SOURCE_PATH = (
    Path(SEED_RAW_SOURCE_PATH).expanduser().resolve() if SEED_RAW_SOURCE_PATH is not None else None
)


In [ ]:
if IN_COLAB and WORKFLOW_CONFIG.enable_data_upload and ACTIVE_TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP and RESOLVED_SEED_RAW_SOURCE_PATH is None:
    from google.colab import files  # type: ignore[import-untyped]
    print("📤 Subí un backup PostgreSQL raw o un CSV de telemetría raw:")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Seleccioná exactamente un archivo .backup o .csv.")
    _uploaded_name = next(iter(uploaded))
    if Path(_uploaded_name).suffix.lower() not in {'.backup', '.csv'}:
        raise RuntimeError("La fuente semilla debe ser un archivo .backup o .csv.")
    _source_destination = DATA_RAW_DIR / Path(_uploaded_name).name
    shutil.move(_uploaded_name, _source_destination)
    RESOLVED_SEED_RAW_SOURCE_PATH = _source_destination.resolve()
    print(f"✅ Fuente semilla explícita: {RESOLVED_SEED_RAW_SOURCE_PATH}")
else:
    if RESOLVED_SEED_RAW_SOURCE_PATH is not None:
        print(f"📂 Fuente semilla declarada: {RESOLVED_SEED_RAW_SOURCE_PATH}")
    elif HITL_CATALOG_PATH.is_file():
        print(f"📚 Catálogo HITL disponible: {HITL_CATALOG_PATH}")
    elif not WORKFLOW_CONFIG.enable_data_upload:
        print("ℹ️ Upload desactivado; la siguiente etapa buscará PostgreSQL.")
    else:
        print("📂 No hay una fuente local disponible.")

# Un pg_dump binario requiere un cliente PostgreSQL compatible del sistema.
PG_RESTORE_PATH: str | None = shutil.which("pg_restore")


In [ ]:
_declared_backup = RESOLVED_SEED_RAW_SOURCE_PATH if RESOLVED_SEED_RAW_SOURCE_PATH is not None and RESOLVED_SEED_RAW_SOURCE_PATH.suffix.lower() == '.backup' else Path(_backup_dest)
_declared_csv = RESOLVED_SEED_RAW_SOURCE_PATH if RESOLVED_SEED_RAW_SOURCE_PATH is not None and RESOLVED_SEED_RAW_SOURCE_PATH.suffix.lower() == '.csv' else Path(_csv_dest)
PG_RESTORE_PATH = resolve_pg_restore_for_backup(
    _declared_backup,
    _declared_csv,
    in_colab=IN_COLAB,
)


In [ ]:
print("➡️ Siguiente paso: cargá y validá las fuentes declaradas.")

In [ ]:
# El preset validado determina el modo y las fuentes compatibles.

SEED_STORE = VersionedSeedStore(SEED_ARTIFACT_ROOT)
seed_snapshot = SEED_STORE.load_current()
HITL_CATALOG_DESCRIPTOR = None
training_db_settings = None
if WORKFLOW_CONFIG.enable_postgres_ingestion:
    training_db_settings = get_optional_database_settings(DatabaseProfile.TRAINING)
    if training_db_settings is None:
        raise RuntimeError(
            "PostgreSQL ingestion is enabled, but the read-only training profile is not configured. "
            "Configure the read-only training profile according to the canonical Colab guide."
        )
if ACTIVE_TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP:
    RAW_SOURCES = []
    if RESOLVED_SEED_RAW_SOURCE_PATH is not None and RESOLVED_SEED_RAW_SOURCE_PATH.suffix.lower() == '.backup':
        RAW_SOURCES.append(
            PostgresBackupSource(RESOLVED_SEED_RAW_SOURCE_PATH, Path(PG_RESTORE_PATH) if PG_RESTORE_PATH else None)
        )
    elif RESOLVED_SEED_RAW_SOURCE_PATH is not None and RESOLVED_SEED_RAW_SOURCE_PATH.suffix.lower() == '.csv':
        RAW_SOURCES.append(RawCsvSource(RESOLVED_SEED_RAW_SOURCE_PATH))
    if training_db_settings is not None:
        RAW_SOURCES.append(PostgresSource(training_db_settings))
    if not RAW_SOURCES:
        raise RuntimeError(
            "No seed source is available. Upload a raw backup/CSV or enable PostgreSQL ingestion."
        )
    SEED_SOURCES = []
    FEEDBACK_SOURCES = []
else:
    RAW_SOURCES = []  # HITL reutiliza features procesadas y no recarga telemetría cruda.
    if seed_snapshot is None:
        raise FileNotFoundError(f"No existe una semilla inmutable activa en {SEED_ARTIFACT_ROOT}. Primero ejecutá SEED_BOOTSTRAP.")
    SEED_SOURCES = [SeedDatasetPackageSource(seed_snapshot.path)]
    FEEDBACK_SOURCES = []
    if HITL_CATALOG_PATH.is_file():
        FEEDBACK_SOURCES.append(
            HitlCatalogSource(HITL_CATALOG_PATH, CatalogSelection.ALL_ACTIVE)
        )
    if training_db_settings is not None:
        FEEDBACK_SOURCES.append(PostgresSource(training_db_settings))
    if not FEEDBACK_SOURCES:
        raise RuntimeError(
            "No HITL feedback source is available. Finalize review packages or enable PostgreSQL ingestion."
        )


In [ ]:

TRAINING_INPUTS = TrainingIngestionPlan(
    mode=ACTIVE_TRAINING_MODE,
    raw_sources=tuple(RAW_SOURCES),
    seed_sources=tuple(SEED_SOURCES),
    feedback_sources=tuple(FEEDBACK_SOURCES),
    feedback_policy=FeedbackPolicy.VALIDATED_ONLY,
)
training_inputs = load_training_inputs(TRAINING_INPUTS)
df_raw = training_inputs.raw
seed_feature_frame = training_inputs.seed_features
validated_feedback = training_inputs.validated_feedback
confirmed_incidents = training_inputs.confirmed_incidents
DATA_SOURCE = ','.join(training_inputs.provenance['source_type'].astype(str))
display(training_inputs.provenance)
_catalog_rows = training_inputs.provenance.loc[training_inputs.provenance['source_type'].eq('HitlCatalogSource')]
if not _catalog_rows.empty:
    _catalog_record = _catalog_rows.iloc[-1].to_dict()
    HITL_CATALOG_DESCRIPTOR = {key: _catalog_record[key] for key in ('contract', 'revision', 'catalog_sha256', 'package_ids', 'package_fingerprints', 'package_sha256', 'resolved_validations', 'duplicate_rows_resolved', 'corrections_resolved') if key in _catalog_record}
for _source in training_inputs.provenance.to_dict(orient='records'):
    if _source.get('archive_table'):
        print(
            f"Detected backup table: {_source['archive_table']} "
            f"({_source['backup_layout']} raw telemetry) | "
            f"reader={_source['reader_version']} | imported rows={_source['rows']}"
        )
print(f"Modo: {ACTIVE_TRAINING_MODE.value}")
print(f"Filas raw: {len(df_raw)} | semilla procesada: {len(seed_feature_frame)} | feedback estable validado: {len(validated_feedback)} | incidentes confirmados: {len(confirmed_incidents)}")
if not df_raw.empty:
    print(f"Rango temporal raw: {df_raw['record_time'].min()} → {df_raw['record_time'].max()}")
print("➡️ Siguiente paso: prepará la augmentación exclusiva del inicio semilla.")


In [ ]:
# Los datos históricos no contienen accidentes confirmados y tienen poco soporte
# de congestión. Las secuencias sintéticas estresan esos límites, pero nunca
# convierten Accident en una salida aprendida del MLP.

if "df_raw" not in globals() or not isinstance(df_raw, pd.DataFrame):
    raise RuntimeError("Primero ejecutá la carga de datos.")
_n_before = len(df_raw)
if ACTIVE_TRAINING_MODE is TrainingMode.HITL_RETRAINING or df_raw.empty:
    _n_synthetic = 0
    print("ℹ️ Datos sintéticos omitidos: sólo se agregan durante el inicio semilla.")
else:
    df_raw = augment_with_synthetic(
        df_raw, n_accident_seq=10, n_congestion_seq=10, records_per_seq=10, seed=RANDOM_SEED
    )
    _n_synthetic = len(df_raw) - _n_before
    print(f"   Zona horaria canónica: {df_raw['record_time'].dt.tz}")

print(f"✅ Augmentación sintética: {_n_synthetic} registros agregados")
print("   Accident: 10 secuencias × 10 = 100 registros de stress técnico")
print("   Congested: 10 secuencias × 10 = 100 registros de entrenamiento")
print(f"   Dataset total: {len(df_raw)} registros ({_n_before} reales + {_n_synthetic} sintéticos)")

if not df_raw.empty:
    origin_summary = summarize_data_origin(df_raw)
    print("\n📋 Procedencia del dataset:")
    display(origin_summary) if "display" in dir() else print(origin_summary.to_string(index=False))
print("➡️ Siguiente paso: calculá o validá las 19 features.")


## 3. Convertir telemetría cruda en 19 features

Los datos raw traen velocidades y conteos. La ingeniería agrega relaciones entre minutos para que el modelo pueda reconocer cambios, acumulación y persistencia.

| Feature | Origen | Qué aporta |
|---|---|---|
| `avg_speed` | Directo | Velocidad media del flujo |
| `total_vehicles` | Directo | Volumen total por minuto |
| Conteos por tipo (5) | Directo | Composición del tránsito |
| `heavy_vehicle_ratio` | Derivado | Proporción de vehículos pesados |
| `delta_speed`, `delta_count` | Derivado | Cambio frente al minuto anterior |
| `transition_flag` | Derivado | Cambio brusco simultáneo de velocidad y volumen |
| `speed_variance` | Derivado | Estabilidad reciente de la velocidad |
| `cumulative_delta_speed` | Derivado | Tendencia acumulada dentro del clip |
| `low_speed_persistence` | Derivado | Cuánto dura una condición lenta |
| Señales de calidad y movimiento (3) | Directo/derivado | Confiabilidad y vehículos detenidos |
| `hour_of_day`, `weather_condition` | Temporal | Contexto horario y ambiental |

El primer registro de cada secuencia no tiene un minuto anterior para calcular diferencias, por eso se descarta de forma controlada.

> **Nota**: el inicio semilla agrega 200 registros sintéticos (100 de congestión y 100 de stress de incidente). Feature engineering los procesa con la misma semántica temporal. Accident se separa antes del target; Congested sintético sólo puede entrar en train con peso reducido.

In [ ]:
# Conservá el orden exacto de las 19 features canónicas requerido por el modelo.

audit_frame = (
    (df_raw if not df_raw.empty else seed_feature_frame)
    if ACTIVE_TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP
    else validated_feedback
)
dataset_audit = audit_training_dataset(audit_frame, require_production_eligible=False)
print("📋 Auditoría previa al entrenamiento:")
print(json.dumps(dataset_audit.report, indent=2, default=str))

engineered_proxy_frame = engineer_features(df_raw) if not df_raw.empty else validated_feedback.head(0).copy()
legacy_missing = [column for column in FEATURE_COLS if not engineered_proxy_frame.empty and engineered_proxy_frame[column].isna().any()]
if legacy_missing:
    print("⚠️ Las filas legacy v1 no contienen evidencia moderna de calidad.")
    print("   Este entrenamiento puede crear un piloto experimental, nunca producción.")
    print(f"   Neutralización conservadora de columnas desconocidas: {legacy_missing}")
    engineered_proxy_frame[legacy_missing] = engineered_proxy_frame[legacy_missing].fillna(0.0)

# El CSV de features aporta reproducibilidad; nunca reemplaza la fuente cruda.
csv_path = os.path.join(DATA_PROCESSED_DIR, "traffic_telemetry.csv")
engineered_proxy_frame.to_csv(csv_path, index=False)

print(f"✅ Features listas: {engineered_proxy_frame.shape[0]} filas × {engineered_proxy_frame.shape[1]} columnas")
print(f"   CSV reproducible → {os.path.abspath(csv_path)}")
print("\n📊 Correlación con avg_speed:")
if not engineered_proxy_frame.empty:
    corr = engineered_proxy_frame[FEATURE_COLS].corr()["avg_speed"].drop("avg_speed").sort_values()
    print(corr.to_string())
else:
    print("Modo HITL sin raw: no corresponde calcular correlación sobre filas raw.")
print("➡️ Siguiente paso: asigná o incorporá las etiquetas.")

## 4. Crear etiquetas provisionales

En el inicio semilla todavía no hay miles de etiquetas humanas. Por eso se usa una matriz de reglas como verdad provisional, calibrada con la distribución observada en el Puente Belgrano.

- **Accident (3)** no es una salida del MLP. Los escenarios sintéticos de incidente se reservan para pruebas técnicas del detector jerárquico.
Los valores concretos no se duplican en Markdown: la siguiente celda imprime la matriz vigente directamente desde `LABELING_THRESHOLDS`, única fuente de verdad.

Los datos sintéticos de Congested sólo pueden aumentar `train`; nunca forman parte de validation/test. Estas reglas son etiquetas proxy y no sustituyen ground truth humano.

**Límite importante:** una etiqueta proxy no es verdad humana. Sólo `vaaet_feedback.human_validations` aporta ground truth; una predicción sin revisar nunca es target. Ver [sesgos y limitaciones](../../../docs/ml/bias-and-limitations.md).

In [ ]:
# El etiquetado proxy usa tres clases estables; Accident nunca es target del MLP.

print("📐 Matriz activa de etiquetas provisionales:")
print(json.dumps(dict(LABELING_THRESHOLDS), indent=2))
scenario = engineered_proxy_frame.get("synthetic_scenario", pd.Series("observed", index=engineered_proxy_frame.index))
incident_stress_frame = engineered_proxy_frame.loc[scenario.eq("accident")].copy()
new_proxy_features = engineered_proxy_frame.loc[~scenario.eq("accident")].copy()
if not new_proxy_features.empty:
    new_proxy_features["traffic_state"] = assign_stable_traffic_state(new_proxy_features)
    new_proxy_features["is_human_validated"] = False
    new_proxy_features["feature_schema_version"] = FEATURE_SCHEMA_VERSION
proxy_frames = [frame for frame in (seed_feature_frame, new_proxy_features) if not frame.empty]
proxy_features = pd.concat(proxy_frames, ignore_index=True) if proxy_frames else validated_feedback.head(0).copy()
if ACTIVE_TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP and not new_proxy_features.empty:
    metadata_columns = [column for column in new_proxy_features if column not in FEATURE_COLS]
    seed_export = new_proxy_features[[*metadata_columns, *FEATURE_COLS]]
    seed_snapshot = SEED_STORE.resolve(
        seed_export,
        SeedArtifactConfig(
            store_root=SEED_ARTIFACT_ROOT,
            action=DatasetArtifactAction(WORKFLOW_CONFIG.seed_artifact_action),
            update_reason=WORKFLOW_CONFIG.seed_artifact_update_reason,
            git_commit=GIT_COMMIT,
            vaaet_version=package_version("vaaet-ml"),
        ),
    )
    print(f"💾 Snapshot semilla inmutable → {seed_snapshot.path}")
    print(json.dumps(seed_snapshot.descriptor, indent=2))
df_features = compose_supervised_dataset(proxy_features, validated_feedback)
if not validated_feedback.empty:
    print(f"✅ Se incorporaron {len(validated_feedback)} etiquetas humanas estables; {len(confirmed_incidents)} incidentes confirmados quedaron fuera del MLP.")

dist = df_features["traffic_state"].value_counts().sort_index()
print("📊 Distribución de estados:")
for code, count in dist.items():
    pct = 100 * count / len(df_features)
    print(f"   {STATE_LABELS[code]:>10} ({code}): {count:>5} records ({pct:.1f}%)")

n_classes = dist.index.nunique()
if n_classes < 2:
    print("🔴 Sólo se encontró una clase: los umbrales no separan este dataset.")
else:
    print(f"\n✅ Se detectaron {n_classes} clases")


In [ ]:

for code, label in {code: STATE_LABELS[code] for code in range(N_MODEL_STATES)}.items():
    if code not in dist.index:
        print(f"⚠️ La clase '{label}' ({code}) no tiene ejemplos y se excluirá.")

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]
bars = ax.bar(
    [STATE_LABELS[c] for c in sorted(dist.index)],
    [dist[c] for c in sorted(dist.index)],
    color=[colors[c] for c in sorted(dist.index)],
)
ax.set_ylabel("Registros")
ax.set_title("Distribución de estados (etiquetado provisional)")
for bar, count in zip(bars, [dist[c] for c in sorted(dist.index)]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

support_summary = summarize_state_balance(df_features)
print("\n📋 Soporte por procedencia:")
display(support_summary) if "display" in dir() else print(support_summary.to_string(index=False))

print("\n📝 Notas sobre el soporte:")
for note in build_class_support_notes(df_features):
    print(f"   - {note}")
print("➡️ Siguiente paso: verificá el feedback humano disponible.")


## 5. Incorporar correcciones humanas (modo HITL)

Las correcciones humanas entran en este punto y prevalecen sobre etiquetas proxy coincidentes. `Accident` confirmado se reserva para evaluar el detector de incidentes y nunca se convierte en target del MLP.


In [ ]:
print(f"Etiquetas humanas estables incluidas: {len(validated_feedback)}")
print(f"Incidentes confirmados reservados fuera del MLP: {len(confirmed_incidents)}")
if validated_feedback.empty:
    print("ℹ️ No hay feedback humano: los gates de producción permanecerán bloqueados.")
print("➡️ Siguiente paso: construí particiones sin leakage.")


## 6. Separar datos y balancear sin leakage

El clasificador aprende únicamente Normal, Reduced y Congested. Accident se excluye del target y se gestiona con la política jerárquica.

1. **Scaler:** normaliza las 19 features usando únicamente train.
2. **Train/validation/test:** mantiene clips completos y reserva los grupos temporales posteriores para test.
3. **Balanceo conservador:** compara alternativas en validation; los sintéticos sólo aparecen en train.

El scaler se exporta como `feature_scaler.joblib` para que inferencia aplique exactamente la misma transformación.

In [ ]:

human_holdout_snapshot = None
if WORKFLOW_CONFIG.human_holdout_frozen:
    human_holdout_snapshot = resolve_human_holdout(
        validated_feedback,
        HumanHoldoutConfig(
            store_root=HUMAN_HOLDOUT_STORE_ROOT,
            action=HumanHoldoutAction(WORKFLOW_CONFIG.human_holdout_action),
            update_reason=WORKFLOW_CONFIG.human_holdout_update_reason,
            validation_size=0.2,
            test_size=0.2,
            random_state=RANDOM_SEED,
            git_commit=GIT_COMMIT,
            vaaet_version=package_version("vaaet-ml"),
        ),
    )
    print(f"🔒 Frozen human holdout: {json.dumps(human_holdout_snapshot.descriptor)}")

partitions = build_training_partitions(
    proxy_features, validated_feedback, ACTIVE_TRAINING_MODE,
    test_size=0.2, validation_size=0.2, random_state=RANDOM_SEED,
    frozen_holdout=human_holdout_snapshot,
)
train_frame = partitions.train
validation_frame = partitions.validation
test_frame = partitions.test

supervision_weight, supervision_report = build_supervision_weights(train_frame, ACTIVE_TRAINING_MODE)
active_supervision = supervision_weight > 0
discarded_proxy_rows = int((~active_supervision).sum())
if discarded_proxy_rows:
    train_frame = train_frame.loc[active_supervision].copy()
    supervision_weight = supervision_weight[active_supervision]
    print(f"ℹ️ Removed {discarded_proxy_rows} expired proxy-memory rows before scaling.")
has_effective_proxy_memory = bool(((~train_frame["is_human_validated"].fillna(False)) & pd.Series(supervision_weight > 0, index=train_frame.index)).any())
if ACTIVE_TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP or has_effective_proxy_memory:
    MODEL_INPUT_POLICY = ModelInputPolicy.LEGACY_V1_BOOTSTRAP
elif set(train_frame['feature_schema_version'].astype(str)) == {'traffic-features-v2'}:
    MODEL_INPUT_POLICY = ModelInputPolicy.CANONICAL_V2
else:
    MODEL_INPUT_POLICY = ModelInputPolicy.CANONICAL_V3
X_train_raw = apply_model_input_policy(train_frame, MODEL_INPUT_POLICY).to_numpy()
X_validation_raw = apply_model_input_policy(validation_frame, MODEL_INPUT_POLICY).to_numpy()
X_test_raw = apply_model_input_policy(test_frame, MODEL_INPUT_POLICY).to_numpy()
y_train = train_frame["traffic_state"].to_numpy(dtype=int)
y_validation = validation_frame["traffic_state"].to_numpy(dtype=int)
y_test = test_frame["traffic_state"].to_numpy(dtype=int)


In [ ]:

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_validation = scaler.transform(X_validation_raw)
X_test = scaler.transform(X_test_raw)

balance_candidates = build_balance_candidates(
    train_frame, supervision_weight, random_state=RANDOM_SEED
)
synthetic_train = train_frame.get("data_origin", pd.Series("real", index=train_frame.index)).eq("synthetic").to_numpy()

print("📊 Partición protegida contra leakage:")
print(f"   Train={len(train_frame)} | Validation={len(validation_frame)} | Test={len(test_frame)}")
print(f"   Alternativas de balanceo: {[strategy.value for strategy in balance_candidates]}")
print(f"   Política de entrada: {MODEL_INPUT_POLICY.value}")
print(f"   Política de supervisión: {json.dumps(supervision_report, default=str)}")
print(f"   Sintéticos en validation/test: {(validation_frame.get('data_origin') == 'synthetic').sum() if 'data_origin' in validation_frame else 0}/{(test_frame.get('data_origin') == 'synthetic').sum() if 'data_origin' in test_frame else 0}")
partition_summary = pd.DataFrame([
    {"partition": name, "records": len(frame), "clips": build_group_ids(frame).nunique(), "start": normalize_timestamp_series(frame["record_time"]).min(), "end": normalize_timestamp_series(frame["record_time"]).max(), "missing_features": int(frame[FEATURE_COLS].isna().sum().sum())}
    for name, frame in (("train", train_frame), ("validation", validation_frame), ("test", test_frame))
])
display(partition_summary) if "display" in dir() else print(partition_summary.to_string(index=False))

print("\nℹ️ El scaler se publicará junto al bundle sólo después de validar todos los artefactos.")

print("✅ Sin SMOTE por defecto: validation y test permanecen reales e intactos.")
print("➡️ Siguiente paso: entrená y compará las alternativas de balanceo.")


## 7. Entrenar el MLP tabular

El modelo es un **MLP** deliberadamente simple. Aprende combinaciones no lineales de las 19 features sin reemplazar la política temporal del pipeline.

- `Dense(64) → Dense(32)`: aprende y comprime relaciones entre features.
- `BatchNormalization` y `Dropout`: estabilizan y reducen sobreajuste.
- `Dense(3, softmax)`: produce Normal, Reduced o Congested. Nunca Accident.

In [ ]:
from uuid import uuid4

n_classes: int = N_MODEL_STATES
n_features: int = X_train.shape[1]
TRAINING_FIT_CONFIG = TrainingFitConfig(random_seed=RANDOM_SEED)
TRAINING_PIPELINE_RUN_ID = str(uuid4())
_numeric_inputs = pd.concat([train_frame, validation_frame, test_frame], ignore_index=True)
_numeric_representations = (
    _numeric_inputs.get('numeric_representation', pd.Series('not-declared', index=_numeric_inputs.index))
    .fillna('not-declared').astype(str).value_counts().sort_index().to_dict()
)
training_input_lock = create_training_input_lock(
    TRAINING_RUNS_ROOT,
    training_pipeline_run_id=TRAINING_PIPELINE_RUN_ID,
    training_mode=ACTIVE_TRAINING_MODE.value,
    seed_snapshot=seed_snapshot.descriptor if seed_snapshot is not None else None,
    hitl_catalog=HITL_CATALOG_DESCRIPTOR,
    human_holdout=human_holdout_snapshot.descriptor if human_holdout_snapshot else None,
    result_rows={"train": len(train_frame), "validation": len(validation_frame), "test": len(test_frame)},
    resolution={
        "validated_feedback": len(validated_feedback),
        "confirmed_incidents": len(confirmed_incidents),
        "discarded_proxy_rows": discarded_proxy_rows,
    },
    numeric_representations=_numeric_representations,
)
print(f"🔐 Input lock sellado antes de entrenar: {training_input_lock.path}")
cross_validation = None
if WORKFLOW_CONFIG.run_grouped_cross_validation:
    cv_source = validated_feedback if ACTIVE_TRAINING_MODE is TrainingMode.HITL_RETRAINING else df_features
    cross_validation = run_grouped_cross_validation(
        cv_source, input_policy=MODEL_INPUT_POLICY, random_seed=RANDOM_SEED,
        model_factory=build_traffic_state_mlp,
        callbacks_factory=lambda: build_training_callbacks(TRAINING_FIT_CONFIG),
        epochs=TRAINING_FIT_CONFIG.epochs, batch_size=TRAINING_FIT_CONFIG.batch_size,
        clear_session=tf.keras.backend.clear_session, set_random_seed=set_keras_random_seed,
    )
    print(f"🔁 Evidencia adicional: F1 cruzado={cross_validation.mean_f1_macro:.4f}")
else:
    print("ℹ️ Validación cruzada adicional desactivada; no cambia los gates.")
print(f"🏗️ MLP: {n_features} features → {n_classes} clases; Accident no se aprende.")
_training_engine = get_engine(training_db_settings) if training_db_settings is not None else None
_training_metadata = PipelineRunMetadata(
    workflow=PipelineWorkflow.TRAINING, application_name="vaaet-ml-training", application_version=package_version("vaaet-ml"), git_commit=GIT_COMMIT,
    source_kind="composed-dataset", input_rows=len(X_train), model_version=MODEL_VERSION,
)


In [ ]:
_training_run, _training_started_at = start_pipeline_run(
    _training_metadata, engine=_training_engine,
    local_manifest_directory=REPO_ROOT / "data/processed/pipeline-runs",
    run_id=TRAINING_PIPELINE_RUN_ID,
)
try:
    _selection = select_balance_candidate(
        candidates=balance_candidates, train_frame=train_frame, x_train=X_train, y_train=y_train,
        x_validation=X_validation, y_validation=y_validation, validation_frame=validation_frame,
        scaler=scaler, input_policy=MODEL_INPUT_POLICY, input_features=n_features,
        output_classes=n_classes, fit_config=TRAINING_FIT_CONFIG,
        callbacks_factory=lambda: build_training_callbacks(TRAINING_FIT_CONFIG),
        clear_session=tf.keras.backend.clear_session, set_random_seed=set_keras_random_seed,
    )
    _training_run.set_output_rows(len(balance_candidates[_selection.strategy].row_positions))
except Exception as _training_error:
    finish_pipeline_run(
        _training_run, status="failed", started_at=_training_started_at,
        engine=_training_engine, local_manifest_directory=REPO_ROOT / "data/processed/pipeline-runs",
        error_category=type(_training_error).__name__,
    )
    if _training_engine is not None:
        dispose_engine(_training_engine)
    raise

balance_selection_report = _selection.report
SELECTED_BALANCE_STRATEGY = _selection.strategy
model, history = _selection.model, _selection.history
sample_weight, class_weights = _selection.sample_weight, _selection.class_weights


In [ ]:

display(balance_selection_report) if "display" in dir() else print(balance_selection_report.to_string(index=False))
print(f"✅ Estrategia elegida: {SELECTED_BALANCE_STRATEGY.value}")
model.summary()

plot_training_history(history.history)

best_epoch = np.argmin(history.history["val_loss"]) + 1
print(f"\n✅ Entrenamiento terminado — mejor época: {best_epoch}")
print("➡️ Siguiente paso: evaluá el candidato sobre test.")

## 8. Evaluar el candidato

La exactitud global puede ocultar fallos en clases pequeñas. Por eso se muestran:

- **F1-macro de tres estados** ≥ 0.88, siempre acompañado por soporte real y número de clips
- **Coste de confusión**: penaliza especialmente los errores directos Normal ↔ Congested
- **ECE**: impide presentar softmax como probabilidad fiable sin comprobar calibración
- **Matriz de confusión**: muestra qué estados se confunden entre sí

Accident no tiene recall publicable sin casos reales. El bundle se marca experimental mientras falte telemetría v2 y holdout humano.

In [ ]:

# La calibración y los umbrales se seleccionan únicamente con validación.
validation_proba_raw = model.predict(X_validation, verbose=0)
temperature = fit_temperature(validation_proba_raw, y_validation)
validation_proba = apply_temperature_scaling(validation_proba_raw, temperature)
decision_policy = select_validation_decision_policy(validation_frame, y_validation, validation_proba, temperature=temperature)
print(f"🎛️ Política elegida exclusivamente con validation: {decision_policy}")

# Test evalúa la cadena productiva exacta sobre grupos reales congelados.
y_proba_raw = model.predict(X_test, verbose=0)
y_proba = apply_temperature_scaling(y_proba_raw, temperature)
classified_test = classify_telemetry_dataframe(test_frame, model, scaler, decision_policy=decision_policy, input_policy=MODEL_INPUT_POLICY)
y_model_pred = y_proba.argmax(axis=1).astype(int)
y_pred = classified_test["traffic_state"].to_numpy(dtype=int)
direct_target_accuracy = float((y_model_pred == y_test).mean())
if ACTIVE_TRAINING_MODE is TrainingMode.SEED_BOOTSTRAP:
    weak_supervision_fidelity = direct_target_accuracy
    human_holdout_direct_accuracy = None
    print(f"Fidelidad a las reglas provisionales (MLP directo): {direct_target_accuracy:.2%}")
else:
    weak_supervision_fidelity = None
    human_holdout_direct_accuracy = direct_target_accuracy
    print(f"Exactitud directa sobre holdout humano: {direct_target_accuracy:.2%}")

present_classes = sorted(np.unique(np.concatenate([y_test, y_pred])))
target_names = [STATE_LABELS[c] for c in present_classes]

print("=" * 60)
print("REPORTE DE CLASIFICACIÓN")
print("=" * 60)
report = classification_report(
    y_test, y_pred,
    labels=present_classes,
    target_names=target_names,
    zero_division=0,
)
print(report)
test_clip_ids = build_group_ids(test_frame).to_numpy()
support_with_intervals = build_classification_support_table(y_test, y_pred, clip_ids=test_clip_ids)
print("\nSoporte por clase (filas y clips):")
display(support_with_intervals) if "display" in dir() else print(support_with_intervals.to_string(index=False))

direct_f1_macro = f1_score(y_test, y_model_pred, labels=[0, 1, 2], average="macro", zero_division=0)
final_f1_macro = f1_score(y_test, y_pred, labels=[0, 1, 2], average="macro", zero_division=0)
direct_confusion_cost = expected_confusion_cost(y_test, y_model_pred)
final_confusion_cost = expected_confusion_cost(y_test, y_pred)
ece = expected_calibration_error(y_test, y_proba)
brier = multiclass_brier_score(y_test, y_proba)
direct_extreme_error = float((((y_test == 0) & (y_model_pred == 2)) | ((y_test == 2) & (y_model_pred == 0))).mean())
final_extreme_error = float((((y_test == 0) & (y_pred == 2)) | ((y_test == 2) & (y_pred == 0))).mean())
direct_intervals = grouped_classification_intervals(y_test, y_model_pred, test_clip_ids, random_state=RANDOM_SEED)
final_intervals = grouped_classification_intervals(y_test, y_pred, test_clip_ids, probabilities=y_proba, random_state=RANDOM_SEED)
print("\nIntervalos agrupados del 95% — salida directa:")
display(direct_intervals) if 'display' in dir() else print(direct_intervals.to_string(index=False))
print("\nIntervalos agrupados del 95% — estado final:")
display(final_intervals) if 'display' in dir() else print(final_intervals.to_string(index=False))


In [ ]:
automatic_accident_states = int(classified_test["traffic_state"].eq(3).sum())
incident_candidate_count = int(classified_test["accident_alert_started"].sum())
reliable_negative_mask = classified_test["measurement_reliable"].fillna(False).astype(bool)
negative_exposure_hours = float(reliable_negative_mask.sum()) / 60.0
reliable_incident_candidates = int((classified_test["accident_alert_started"] & reliable_negative_mask).sum())
false_candidates_per_hour = reliable_incident_candidates / negative_exposure_hours if negative_exposure_hours else float("nan")
false_candidates_upper_95 = false_alert_rate_upper_bound(reliable_incident_candidates, negative_exposure_hours) if negative_exposure_hours else None
if automatic_accident_states != 0:
    raise RuntimeError("Se violó una garantía: Accident fue producido automáticamente.")
synthetic_incident_episode_sensitivity = None
if not incident_stress_frame.empty:
    stress = incident_stress_frame.copy()
    stress["traffic_state"] = 2
    stress["state_label"] = STATE_LABELS[2]
    stress["confidence"] = 0.5
    stress = apply_conservative_accident_gate(stress)
    episode_hits = stress.groupby("clip_id")["accident_alert_started"].any()
    synthetic_incident_episode_sensitivity = float(episode_hits.mean())
    print(f"Sensibilidad sobre stress sintético de incidentes: {synthetic_incident_episode_sensitivity:.2%} (prueba técnica, no recall real)")
confirmed_incident_candidate_sensitivity = None
confirmed_incident_support = len(confirmed_incidents)
if confirmed_incident_support:
    human_incident_context = pd.concat([validated_feedback, confirmed_incidents], ignore_index=True).sort_values(["clip_id", "record_time"])
    human_incident_context["traffic_state"] = 2
    human_incident_context["state_label"] = STATE_LABELS[2]
    human_incident_context["confidence"] = 0.5
    human_incident_context = apply_conservative_accident_gate(human_incident_context)
    confirmed_keys = set(zip(confirmed_incidents["clip_id"], normalize_timestamp_series(confirmed_incidents["record_time"])))
    evaluated_keys = list(zip(human_incident_context["clip_id"], normalize_timestamp_series(human_incident_context["record_time"])))
    confirmed_mask = pd.Series([key in confirmed_keys for key in evaluated_keys], index=human_incident_context.index)
    confirmed_incident_candidate_sensitivity = float(human_incident_context.loc[confirmed_mask, "accident_rule_triggered"].mean())
    print(f"Detección candidata sobre incidentes confirmados: {confirmed_incident_candidate_sensitivity:.2%} (soporte={confirmed_incident_support}; todavía no es recall operacional)")
else:
    print("Detección de incidentes confirmados: sin soporte (0 casos humanos)")
print(f"{'F1 directo':>20}: {direct_f1_macro:.4f}")
print(f"{'F1 final':>20}: {final_f1_macro:.4f}")
print(f"{'Coste directo':>20}: {direct_confusion_cost:.4f}")
print(f"{'Coste final':>20}: {final_confusion_cost:.4f}")
print(f"{'ECE':>15}: {ece:.4f}")
print(f"{'Brier':>15}: {brier:.4f}")
print(f"{'N↔C directo':>20}: {direct_extreme_error:.2%}")
print(f"{'N↔C final':>20}: {final_extreme_error:.2%}")
print(f"{'Candidatos/hora':>15}: {false_candidates_per_hour:.6f} sobre {negative_exposure_hours:.2f} h")
if negative_exposure_hours < 300:
    print("⚠️ La tasa de falsas alertas es preliminar; se necesitan unas 300 horas negativas.")

if final_f1_macro >= 0.88:
    print("✅ F1-macro cumple el objetivo (≥ 0.88)")
else:
    print("⚠️ F1-macro no alcanza 0.88; el bundle continúa experimental.")

print("➡️ Siguiente paso: revisá matrices y confiabilidad.")


### 8.1 Entender dónde se equivoca

Las matrices comparan la salida directa del MLP con el estado final después de umbrales e histéresis. El diagrama de confiabilidad muestra si una confianza alta realmente coincide con una mayor tasa de aciertos.

In [ ]:
plot_training_evaluation(
    y_test, y_model_pred, y_pred, y_proba, state_labels=STATE_LABELS
)

print("\n📊 Recall por estado:")
for row in support_with_intervals.itertuples(index=False):
    status = "✅" if row.recall > 0 else "🔴"
    print(f"   {status} {row.state_label:>10}: {row.recall:.4f} (soporte={row.support})")

print("➡️ Siguiente paso: calculá elegibilidad y exportá el bundle.")


## 9. Decidir promoción y exportar el bundle

Esta etapa no aprueba un modelo por intuición. Reúne blockers, genera el input lock, escribe el manifiesto y deja claro si el resultado es `pilot`, `candidate` o `production`.

In [ ]:
model_path = os.path.join(MODEL_DIR, "traffic_classifier.keras")
label_path = os.path.join(MODEL_DIR, "label_mapping.joblib")
human_test_only = bool(
    "is_human_validated" in test_frame and test_frame["is_human_validated"].fillna(False).all()
)
human_holdout = bool(human_test_only and human_holdout_snapshot is not None)
eligibility = evaluate_candidate_eligibility(
    training_mode=ACTIVE_TRAINING_MODE, dataset_blockers=dataset_audit.blockers,
    human_holdout=human_holdout, test_frame=test_frame, actual=y_test, predicted=y_pred,
    direct_predicted=y_model_pred, f1_macro=final_f1_macro,
    direct_normal_congested_error=direct_extreme_error, final_normal_congested_error=final_extreme_error,
    expected_calibration_error=ece, negative_exposure_hours=negative_exposure_hours,
    false_candidates_per_hour=false_candidates_per_hour, direct_intervals=direct_intervals,
    final_intervals=final_intervals, false_candidates_upper_95=false_candidates_upper_95,
    false_candidate_count=reliable_incident_candidates,
)
metric_gates = dict(eligibility.metric_gates)
promotion_blockers = list(eligibility.promotion_blockers)
production_eligible = eligibility.production_eligible
training_lifecycle = build_training_lifecycle(
    ACTIVE_TRAINING_MODE, MODEL_INPUT_POLICY, production_eligible=production_eligible
)
print(f"Elegibilidad calculada: {production_eligible}; la promoción sigue siendo humana.")


### 9.1 Sellar procedencia y escribir el manifiesto

El comprobante exacto de inputs ya fue sellado antes del cómputo. Esta celda escribe el manifiesto y muestra por qué el modelo quedó como piloto, candidato o producción.

In [ ]:
try:
    bundle_manifest = build_and_publish_bundle(
    MODEL_DIR, model=model, scaler=scaler, label_mapping=dict(STATE_LABELS),
    metrics={"direct_f1_macro": float(direct_f1_macro), "final_f1_macro": float(final_f1_macro), "weak_supervision_fidelity": weak_supervision_fidelity, "human_holdout_direct_accuracy": human_holdout_direct_accuracy, "direct_expected_confusion_cost": direct_confusion_cost, "final_expected_confusion_cost": final_confusion_cost, "ece": ece, "brier_score": brier, "direct_normal_congested_error": direct_extreme_error, "final_normal_congested_error": final_extreme_error, "grouped_direct_intervals": direct_intervals.to_dict(orient="records"), "grouped_final_intervals": final_intervals.to_dict(orient="records"), "automatic_accident_states": automatic_accident_states, "congested_minutes": eligibility.congested_minutes, "congested_clips": eligibility.congested_clips, "incident_candidate_count": reliable_incident_candidates, "negative_exposure_hours": negative_exposure_hours, "false_candidates_per_hour": false_candidates_per_hour, "false_candidates_upper_95": false_candidates_upper_95, "synthetic_incident_episode_sensitivity": synthetic_incident_episode_sensitivity, "confirmed_incident_support": confirmed_incident_support, "confirmed_incident_candidate_sensitivity": confirmed_incident_candidate_sensitivity, "selected_balance_strategy": SELECTED_BALANCE_STRATEGY.value, "balance_validation_candidates": balance_selection_report.to_dict(orient="records"), "production_eligible": production_eligible},
    data_provenance={
        "origin": "training-notebook",
        "dataset_source": str(DATA_SOURCE),
        "record_count": int(len(df_features)),
        "real_record_count_before_engineering": int(_n_before),
        "synthetic_record_count_before_engineering": int(_n_synthetic),
        "synthetic_data_included": bool(synthetic_train.any()),
        "synthetic_records_in_validation": 0,
        "synthetic_records_in_test": 0,
        "telemetry_v3_coverage": dataset_audit.report["telemetry_v3_coverage"],
        "human_test_only": human_test_only,
        "human_holdout": human_holdout,
        "production_eligible": production_eligible,
        "promotion_blockers": promotion_blockers,
    },
    decision_policy=decision_policy,
    training_lifecycle=training_lifecycle,
    human_holdout=human_holdout_snapshot.descriptor if human_holdout_snapshot is not None else None,
        training_input_lock=training_input_lock.descriptor,
    )
except Exception as _export_error:
    finish_pipeline_run(
        _training_run, status="failed", started_at=_training_started_at,
        engine=_training_engine, local_manifest_directory=REPO_ROOT / "data/processed/pipeline-runs",
        error_category=type(_export_error).__name__,
    )
    if _training_engine is not None:
        dispose_engine(_training_engine)
    raise
_training_run.set_model_revision(str(bundle_manifest['model_revision']))
finish_pipeline_run(
    _training_run, status="succeeded", started_at=_training_started_at,
    engine=_training_engine, local_manifest_directory=REPO_ROOT / "data/processed/pipeline-runs",
)
if _training_engine is not None:
    dispose_engine(_training_engine)
print(f"\nEtapa del modelo: {training_lifecycle['deployment_stage'].upper()} | revision={bundle_manifest['model_revision']}")
for blocker in promotion_blockers:
    print(f"   - {blocker}")

print(f"\n💾 Artefactos exportados:")
print(f"   Modelo   → {os.path.abspath(model_path)} ({os.path.getsize(model_path) / 1024:.1f} KB)")
print(f"   Etiquetas → {os.path.abspath(label_path)} (clases: {list(STATE_LABELS.values())})")
print(f"   Scaler → {os.path.abspath(os.path.join(MODEL_DIR, 'feature_scaler.joblib'))}")

print("\n📝 Notas sobre el soporte:")
for note in build_class_support_notes(df_features):
    print(f"   - {note}")

## 10. Informe inmutable de la corrida

El informe se genera sólo después de validar el bundle y los gates existentes. Conserva agregados, fingerprints y decisiones; no incluye filas, videos, notas HITL, secretos ni rutas privadas.

- `training-observability-report.json` es la fuente canónica y el resumen Markdown facilita la revisión humana.
- Los diagnósticos muestran optimización, calidad/calibración y progreso de supervisión.
- La validación cruzada opcional ya se ejecutó antes del entrenamiento final y sólo aporta evidencia adicional.

In [ ]:
report_false_candidates = (
    float(false_candidates_per_hour) if np.isfinite(false_candidates_per_hour) else None
)
training_report = build_training_run_report(
    training_input_lock=training_input_lock, training_lifecycle=training_lifecycle,
    fit_config=TRAINING_FIT_CONFIG, supervision_report=supervision_report,
    partition_rows={"train": len(train_frame), "validation": len(validation_frame), "test": len(test_frame)},
    selected_balance_strategy=SELECTED_BALANCE_STRATEGY.value,
    balance_candidates=balance_selection_report.to_dict(orient="records"),
    training_history=history,
    direct_metrics={"f1_macro": float(direct_f1_macro), "expected_confusion_cost": direct_confusion_cost, "normal_congested_error": direct_extreme_error},
    policy_metrics={"f1_macro": float(final_f1_macro), "expected_confusion_cost": final_confusion_cost, "normal_congested_error": final_extreme_error, "ece": ece, "brier_score": brier},
    incident_metrics={"negative_exposure_hours": negative_exposure_hours, "false_candidates_per_hour": report_false_candidates, "synthetic_incident_episode_sensitivity": synthetic_incident_episode_sensitivity, "confirmed_incident_support": confirmed_incident_support, "confirmed_incident_candidate_sensitivity": confirmed_incident_candidate_sensitivity},
    support_table=support_with_intervals,
    evaluation_evidence=build_training_evaluation_evidence(y_test, y_model_pred, y_pred, y_proba),
    eligibility=eligibility, decision_policy=decision_policy,
    runtime=build_training_runtime_evidence(
        RUNTIME, tensorflow_version=tf.__version__,
        keras_version=str(getattr(tf.keras, "__version__", "unknown")),
        declared_extras=("core-inference", "ml-training", "ml-visualization", "ml-database"),
    ),
    model_version=MODEL_VERSION,
    cross_validation=cross_validation.report_evidence() if cross_validation is not None else None,
)
if WORKFLOW_CONFIG.write_training_report:
    persisted_training_report = write_training_run_report(TRAINING_RUNS_ROOT, training_report)
    diagnostic_paths = save_training_run_diagnostics(persisted_training_report, TRAINING_RUNS_ROOT)
    print(f"📋 Informe inmutable: {persisted_training_report.path}")
    print(f"📈 Diagnósticos generados: {len(diagnostic_paths)}")
else:
    persisted_training_report = None
    print("ℹ️ Informe inmutable desactivado por la configuración validada.")


In [ ]:
if persisted_training_report is not None and WORKFLOW_CONFIG.reference_training_run_id is not None:
    reference_report = load_training_run_report(
        TRAINING_RUNS_ROOT / WORKFLOW_CONFIG.reference_training_run_id / TRAINING_OBSERVABILITY_REPORT_FILE
    )
    report_comparison = compare_training_run_reports(persisted_training_report, reference_report)
    if report_comparison.comparable:
        print(f"📊 Comparación compatible con {report_comparison.reference_run_id}:")
        for metric_name, delta in report_comparison.metric_deltas.items():
            print(f"   {metric_name}: {delta:+.4f}")
    else:
        print("ℹ️ Referencia no comparable; no se calcularon deltas.")
        for reason in report_comparison.reasons:
            print(f"   - {reason}")
else:
    print("ℹ️ Sin referencia compatible solicitada para esta corrida.")


In [ ]:
print("➡️ Revisá el resumen, los tres gráficos y los bloqueos antes de una decisión humana.")
print("ℹ️ El informe y la comparación nunca promueven ni reemplazan el bundle automáticamente.")

## 11. Copiar el bundle a Google Drive

En Colab, esta etapa copia los cuatro archivos terminados a Drive para que inferencia pueda cargarlos después de reiniciar el runtime. En local se omite sin modificar artefactos.

In [ ]:
# La copia se prepara y valida antes del reemplazo atómico en Drive.

if not IN_COLAB:
    print("ℹ️ Entorno local: el bundle permanece en el directorio de artefactos.")
elif not WORKFLOW_CONFIG.copy_bundle_to_drive:
    print("ℹ️ Copia a Drive desactivada; el bundle validado queda en /content.")
else:
    _bundle_directory = Path(MODEL_DIR)
    try:
        from google.colab import drive  # type: ignore[import-untyped]

        drive.mount("/content/drive", force_remount=False)
        _drive_destination = Path("/content/drive") / DRIVE_ARTIFACT_DIR
        publish_bundle_copy(_bundle_directory, _drive_destination)
    except Exception:
        # Drive no ofrece errores estables; detenemos el workflow sin exponer detalles.
        raise RuntimeError("No se pudo copiar el bundle validado a Drive; verificá el montaje y repetí la copia.") from None
    print(f"✅ Bundle validado copiado a Drive: {_drive_destination}")

## 12. Límite de persistencia operacional

El perfil `training` es estrictamente read-only. Este notebook genera artefactos locales/Drive/DVC y no escribe predicciones operacionales. La persistencia pertenece exclusivamente al workflow de inferencia.

La base operativa se organiza en `vaaet_raw.traffic_data`, `vaaet_ml.telemetry_features`, `vaaet_ml.traffic_predictions` y `vaaet_feedback.human_validations`. Las validaciones humanas son append-only y nunca son modificadas por un reentrenamiento.

In [ ]:
print("✅ Entrenamiento terminado sin escrituras en la base operacional.")
print(f"   Artefactos: {os.path.abspath(MODEL_DIR)}")
print("   Siguiente paso: usá analyze_traffic_video.ipynb para probar el bundle y generar feedback.")
